In [ ]:
from dvc.repo import Repo as DVCRepo
import pathlib

r = DVCRepo(r"D:\BEHAVIOR-1K\asset_pipeline")

In [ ]:
# Boolean flag determining whether or not stages will be checked for status
CHECK_PROCESSED = True

In [ ]:
stages = r.stages

In [ ]:
dir(next(s for s in stages if s.is_data_source))
print(pathlib.Path(next(s for s in stages if s.is_data_source).outs[0].fs_path).name)

In [ ]:
s_to_deps = {s: [d.fs_path for d in s.deps] for s in stages}
out_to_s = {o.fs_path: s for s in stages for o in s.outs}

In [ ]:
import collections

out_to_s_dup = collections.defaultdict(list)
for s in stages:
    for o in s.outs:
        out_to_s_dup[o.fs_path].append(s)

In [ ]:
[x for x, v in out_to_s_dup.items() if len(v) > 1]

In [ ]:
import networkx as nx

In [ ]:
G = nx.DiGraph()

In [ ]:
def name(s):
    if s.is_data_source:
        return pathlib.Path(s.outs[0].fs_path).name
    return s.name

In [ ]:
def canonicalize(s):
    if s.is_data_source:
        return pathlib.Path(s.outs[0].fs_path).name

    return s.name.split("@")[0] if not s.is_data_source else "data_source"

In [ ]:
if CHECK_PROCESSED:
    r.lock.lock()

# Process nodes
for s in s_to_deps.keys():
    this_node = canonicalize(s)
    if this_node not in G.nodes:
        G.add_node(this_node, total=0, changed=0, total_set=set(), changed_set=set())

    if CHECK_PROCESSED:
        G.nodes[this_node]["total"] += 1
        G.nodes[this_node]["total_set"].add(name(s))
        if s.changed():
            G.nodes[this_node]["changed"] += 1
            G.nodes[this_node]["changed_set"].add(name(s))

# Add dependencies
for s in s_to_deps.keys():
    this_node = canonicalize(s)
    for dep in s_to_deps[s]:
        if dep in out_to_s:
            from_node = canonicalize(out_to_s[dep])
            G.add_edge(from_node, this_node)

if CHECK_PROCESSED:
    r.lock.unlock()

In [ ]:
for x in nx.topological_sort(G):
    print(x)

In [ ]:
# In mermaid format for pasting into README
for f, t in G.edges:
    print(f"    {f} --> {t}")

In [ ]:
import matplotlib.pyplot as plt
from networkx.drawing.nx_pydot import graphviz_layout

pos = graphviz_layout(G, prog="dot")
nx.draw(G, pos)
plt.show()

In [ ]:
if CHECK_PROCESSED:
    print("Completion ratios:")
    for x in nx.topological_sort(G):
        completion_ratio = 1 - (G.nodes[x]["changed"] / G.nodes[x]["total"])
        completion_percentage = int(completion_ratio * 100)
        print(f"{x}: {completion_percentage}%")

In [ ]:
completed_meshes = (
    G.nodes["export_meshes"]["total_set"] - G.nodes["export_meshes"]["changed_set"]
)
for e in sorted(completed_meshes):
    obj = e.split("/")[-1]
    print(f'    "{obj}",')

In [ ]:
# Clear out the stuff that failed
import ruamel.yaml
import glob, json

with open(r"D:\BEHAVIOR-1K\asset_pipeline\dvc.lock") as f:
    yaml = ruamel.yaml.YAML(typ="rt")
    yaml.default_flow_style = False
    lock = yaml.load(f)

stages = list(lock["stages"].keys())
for stage in stages:
    if "objects/" in stage and "batch" not in stage and "relevant" not in stage:
        del lock["stages"][stage]

with open(r"D:\BEHAVIOR-1K\asset_pipeline\dvc.lock", "w") as f:
    yaml.dump(lock, f)